In [ ]:
import pandas as pd

# load corpus

df = pd.read_csv(
    "elmundo_corpus.csv",
    low_memory=False
)

print("=" * 80)

print("rows")

print(len(df))

print("\ncolumns")

for c in df.columns:
    print(c)

print("\n" + "=" * 80)

# missing values

summary = []

for col in df.columns:

    missing = df[col].isna().sum()

    empty = 0

    if df[col].dtype == object:

        empty = (

            df[col]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()

        )

    summary.append({

        "column": col,
        "missing": missing,
        "empty_strings": empty,
        "filled": len(df) - max(missing, empty)

    })

summary = pd.DataFrame(summary)

print(summary.sort_values("missing", ascending=False))

print("\n" + "=" * 80)

# important fields

important = [

    "url",
    "published_time",
    "title",
    "text",
    "image_urls",
    "image_count",
    "section"

]

print("\nimportant fields\n")

for field in important:

    if field in df.columns:

        print(f"{field:20} OK")

    else:

        print(f"{field:20} MISSING")

In [ ]:
# filter articles about russia

russia_keywords = [

    r"\brusi(a|o|os|as)?\b",
    r"\bkreml(in)?\b",
    r"\bmosc[uú]\b",
    r"\bputin\b",
    r"\blavrov\b",
    r"\bpeskov\b",
    r"\bshoigu\b",
    r"\bgerasimov\b",
    r"\bmedv[eé]dev\b",
    r"\bzakharova\b",
    r"\bmishustin\b",
    r"\bgazprom\b",
    r"\brosneft\b",
    r"\brosatom\b",
    r"\btransneft\b",
    r"\blukoil\b",
    r"\bduma\b",
    r"\bcrimea\b",
    r"\bguerra de ucrania\b",
    r"\bguerra de agresi[oó]n\b"

]

pattern = "|".join(russia_keywords)

russia_df = df[

    (

        df["title"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False
        )

    )

    |

    (

        df["text"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False
        )

    )

].copy()

print(len(russia_df))

In [ ]:
# count articles matched by each keyword

keyword_counts = {}

text = (

    df["title"]
    .fillna("")
    + " "
    + df["text"]
    .fillna("")

)

for keyword in russia_keywords:

    keyword_counts[keyword] = (

        text
        .str.contains(
            keyword,
            case=False,
            na=False
        )
        .sum()

    )

keyword_counts = (

    pd.Series(keyword_counts)
    .sort_values(ascending=False)
    .reset_index()

)

keyword_counts.columns = [

    "keyword",
    "articles"

]

display(keyword_counts)

In [ ]:
# compare filtered corpus with old high recall corpus

old_df = pd.read_csv(
    "elmundo_ukraine_russia_high_recall.csv"
)

old_urls = set(old_df["url"])

new_urls = set(russia_df["url"])

print(f"old corpus: {len(old_urls):,}")

print(f"new corpus: {len(new_urls):,}")

In [ ]:
# compare overlap

print(f"overlap: {len(old_urls & new_urls):,}")

print(f"only old: {len(old_urls - new_urls):,}")

print(f"only new: {len(new_urls - old_urls):,}")

In [ ]:
# save final corpus after the second keyword filtering

russia_df.to_csv(

    "elmundo_SECONDROUND_final_articles.csv",

    index=False,

    encoding="utf-8-sig"

)

print("\nfinal corpus saved")

print(len(russia_df))

In [ ]:
import hashlib
import pandas as pd

INPUT_FILE = (
    "elmundo_SECONDROUND_final_articles.csv"
)

OUTPUT_FILE = (
    "elmundo_SECONDROUND_final_articles.csv"
)

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

filenames = []

for urls in df["image_urls"].fillna(""):

    article_files = []

    for url in str(urls).split("|"):

        url = url.strip()

        if not url.startswith("http"):
            continue

        if "smetrics.el-mundo.net" in url:
            continue

        if "tradedoubler.com" in url:
            continue

        if "i.ytimg.com" in url:
            continue

        if "/graficos/" in url:
            continue

        filename = (
            hashlib.md5(
                url.encode("utf-8")
            ).hexdigest()
            + ".jpg"
        )

        article_files.append(filename)

    filenames.append(
        "|".join(article_files)
    )

df["image_filenames"] = filenames

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("done")

print(df[[
    "image_urls",
    "image_filenames"
]].head())

In [ ]:
import pandas as pd

df = pd.read_csv(
    "elmundo_SECONDROUND_final_articles.csv",
    low_memory=False
)

print(df["image_urls"].iloc[0])

In [ ]:
import os

SOURCE_FOLDER = (
    "."
)

print(os.path.exists(SOURCE_FOLDER))

files = os.listdir(SOURCE_FOLDER)

print(len(files))

print(files[:20])

In [ ]:
import hashlib

url = (
    "https://phantom-elmundo.unidadeditorial.es/"
    "a85e36e3cf4d10bf6c29bcd260f6d195/"
    "resize/1200/f/jpg/assets/multimedia/imagenes/2022/01/01/16410596759431.jpg"
)

print(hashlib.md5(url.encode("utf-8")).hexdigest() + ".jpg")

In [ ]:
import os
import shutil
import hashlib
import pandas as pd

from tqdm import tqdm

# paths

ARTICLES_FILE = (
    "elmundo_SECONDROUND_final_articles.csv"
)

SOURCE_FOLDER = (
    "."
)

OUTPUT_FOLDER = (
    "elmundo_images_SECONDROUND"
)

MISSING_FILE = (
    "elmundo_missing_images.csv"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

# load articles

df = pd.read_csv(
    ARTICLES_FILE,
    low_memory=False
)

# collect image filenames

filenames = set()

for urls in df["image_urls"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if not url.startswith("http"):
            continue

        # skip trackers

        if "smetrics.el-mundo.net" in url:
            continue

        if "tradedoubler.com" in url:
            continue

        # skip youtube previews

        if "i.ytimg.com" in url:
            continue

        # skip graphics

        if "/graficos/" in url:
            continue

        filename = (
            hashlib.md5(
                url.encode("utf-8")
            ).hexdigest()
            + ".jpg"
        )

        filenames.add(
            filename
        )

print(f"articles: {len(df):,}")
print(f"unique images needed: {len(filenames):,}")

# copy images

copied = 0

already_exists = 0

missing = []

for filename in tqdm(
    sorted(filenames)
):

    source = os.path.join(
        SOURCE_FOLDER,
        filename
    )

    destination = os.path.join(
        OUTPUT_FOLDER,
        filename
    )

    if not os.path.exists(
        source
    ):

        missing.append(
            filename
        )

        continue

    if os.path.exists(
        destination
    ):

        already_exists += 1

        continue

    shutil.copy2(
        source,
        destination
    )

    copied += 1

# save missing

pd.DataFrame({

    "filename": missing

}).to_csv(

    MISSING_FILE,

    index=False,

    encoding="utf-8-sig"

)

print()

print(f"copied: {copied:,}")

print(f"already existed: {already_exists:,}")

print(f"missing: {len(missing):,}")

print()

print("saved folder:")

print(OUTPUT_FOLDER)

print()

print("missing file:")

print(MISSING_FILE)

In [ ]:
import os
import time
import hashlib
import requests
import pandas as pd

from tqdm import tqdm

# paths

INPUT_FILE = (
    "elmundo_SECONDROUND_final_articles.csv"
)

OUTPUT_FOLDER = (
    "elmundo_images_SECONDROUND"
)

FAILED_FILE = (
    "elmundo_failed_secondround.csv"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}

MAX_RETRIES = 3

# load articles

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

# collect urls

image_urls = []

for urls in df["image_urls"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if not url.startswith("http"):
            continue

        if "smetrics.el-mundo.net" in url:
            continue

        if "tradedoubler.com" in url:
            continue

        if "i.ytimg.com" in url:
            continue

        if "/graficos/" in url:
            continue

        image_urls.append(url)

image_urls = list(dict.fromkeys(image_urls))

print(f"images to check: {len(image_urls):,}")

downloaded = 0

already_exists = 0

failed = []

for url in tqdm(image_urls):

    filename = (
        hashlib.md5(
            url.encode("utf-8")
        ).hexdigest()
        + ".jpg"
    )

    filepath = os.path.join(
        OUTPUT_FOLDER,
        filename
    )

    if os.path.exists(filepath):

        already_exists += 1
        continue

    success = False

    for _ in range(MAX_RETRIES):

        try:

            response = requests.get(
                url,
                headers=HEADERS,
                timeout=60
            )

            if response.status_code == 200:

                with open(filepath, "wb") as f:

                    f.write(response.content)

                downloaded += 1

                success = True

                break

        except Exception:

            pass

        time.sleep(2)

    if not success:

        failed.append({

            "image_url": url,
            "filename": filename

        })

pd.DataFrame(
    failed
).to_csv(

    FAILED_FILE,

    index=False,

    encoding="utf-8-sig"

)

print()

print(f"downloaded: {downloaded:,}")

print(f"already existed: {already_exists:,}")

print(f"failed: {len(failed):,}")

print()

print("failed file:")

print(FAILED_FILE)

In [ ]:
import pandas as pd

failed = pd.read_csv(
    "elmundo_failed_secondround.csv"
)

print(len(failed))

display(failed.head(20))

In [ ]:
import pandas as pd
import requests

from tqdm import tqdm

failed = pd.read_csv(
    "elmundo_failed_secondround.csv"
)

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}

statuses = []

for url in tqdm(failed["image_url"]):

    try:

        r = requests.get(
            url,
            headers=HEADERS,
            timeout=30,
            allow_redirects=True
        )

        statuses.append({

            "status": r.status_code,
            "url": url

        })

    except Exception as e:

        statuses.append({

            "status": type(e).__name__,
            "url": url

        })

status_df = pd.DataFrame(statuses)

print(status_df["status"].value_counts())



In [ ]:
import pandas as pd
df = pd.read_csv(
    "elmundo_SECONDROUND_final_articles.csv",
    low_memory=False
)
df

In [ ]:
dataset = df.rename(columns={
    "section": "category",
    "published_time": "publish_date",
    "image_urls": "image_url",
    "image_filenames": "image_filename"
})

# длина текста
dataset["text_length"] = (
    dataset["text"]
    .fillna("")
    .str.len()
)

dataset = dataset[
    [
        "category",
        "url",
        "title",
        "publish_date",
        "text",
        "text_length",
        "image_url",
        "image_filename",
    ]
]


print(dataset.shape)
display(dataset.head())

In [ ]:
dataset.to_csv(
    "elmundo_SECONDROUND_final_articles.csv",
    index=False,
    encoding="utf-8-sig"
)
